# 👗 Fashion Image Search
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/fashion-image-search/training/notebook.ipynb)

Upload a photo of a clothing item, a shoe or an accessory and get the most visually similar products
of an 8,000-item fashion catalog. A frozen ImageNet **MobileNetV2** turns every photo into a 1280-d
embedding; an exact **cosine nearest-neighbour** search over the catalog embeddings returns the matches.
Nothing is fine-tuned: "training" = embedding the catalog, comparing representations on held-out
query photos (Precision@K), and exporting the best one as the Hugging Face model repo that the
[Space](https://huggingface.co/spaces/shalev396/fashion-image-search) serves.

All logic lives in `src/` and `../model/model.py` (the same file the Space runs); this notebook calls it step by step.

**Contents:** 1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "fashion-image-search"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has; nothing platform-specific is installed or special-cased.
import tensorflow as tf
device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print("device:", device)

In [ ]:
import time

import pandas as pd
from IPython.display import Image as ShowImage, display

from src import data_setup, engine, export, model_builder, utils   # importing src puts ../model on sys.path
from src.config import Config
import model as M                                                  # ../model/model.py

print(utils.lib_versions("tensorflow", "keras", "numpy", "pandas", "sklearn"))
print("TensorFlow device:", utils.device_name("tensorflow"))

## 2. Config
Every hyperparameter lives in `src/config.py`: an 8,000-product catalog, 500 validation + 500 test
queries, 224x224 inputs, batch 64, PCA sizes 256/128, Precision@1/5/10. `SMOKE_TEST=1` shrinks the run to
a 300-product catalog and 40+40 queries and exports to `outputs/smoke/model` instead of `../model`.

In [ ]:
import os

os.environ.setdefault("SMOKE_TEST", "0")   # "1" = quick smoke run; a SMOKE_TEST set outside the notebook wins
PUSH_TO_HUB = False                         # upload the exported model folder to the Hub (section 8)

cfg = Config()
utils.set_seeds(cfg.seed, "tensorflow")
print("export to:", cfg.model_dir)
cfg

## 3. Data
[Fashion Product Images (Small)](https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small):
44k Myntra product photos (60x80 px) with `styles.csv` labels. kagglehub downloads it once (anonymous);
`styles.csv` is cached in `training/data/`, the photos stay in the kagglehub cache. If Kaggle is unreachable
the code falls back to CIFAR-10 (documented in `src/data_setup.py`).

The split is stratified by master category: the **catalog** gets indexed, **val** queries choose the
variant, **test** queries are reported once. Query photos are never in the catalog.

In [ ]:
df, source = data_setup.load_metadata(cfg)
splits = data_setup.split(cfg, df, source)
data_setup.label_table(splits, "master_category")

In [ ]:
print("article types in the catalog:", splits.index["article_type"].nunique())
print(splits.index["article_type"].value_counts().head(10).to_string())
display(ShowImage(filename=str(export.plot_samples(splits, cfg.outputs_dir / "samples.png"))))

In [ ]:
t0 = time.perf_counter()
images = {name: data_setup.load_images(part["image_path"]) for name, part in splits.items()}
print({name: len(imgs) for name, imgs in images.items()}, "photos decoded; size of the first:", images["index"][0].size)

## 4. Load model
`model.build_embedder` (in `../model/model.py`): raw RGB -> `Rescaling(1/127.5, -1)` -> MobileNetV2 without
its classifier, global average pooling -> 1280-d. ImageNet weights are downloaded once (~9 MB) and stay frozen.
The raw-pixel and colour-histogram baselines in `src/model_builder.py` need no network at all.

In [ ]:
embedder = model_builder.build_embedder(cfg)
params = model_builder.count_params(embedder)
print(f"{embedder.name}: {params:,} parameters (all frozen), output {embedder.output_shape}")
for name in model_builder.variant_names(cfg):
    print(f"  {name:18s} {model_builder.describe(name)}")

## 5. Training
There is no gradient descent: "training" builds each variant's index.
1. Embed catalog, val and test photos with MobileNetV2 (`model.backbone_features`, the same code the Predictor runs).
2. Build every variant: baselines, the full embedding, and PCA versions fit on the catalog embeddings.
   Catalog vectors are stored as float16, exactly like the export.
3. Precision@K curves on the **validation** queries (the "training curves" of a retrieval model).

In [ ]:
raw, embed_time_s = engine.embed_splits(cfg, embedder, images)
print(f"embedding took {embed_time_s:.1f} s")

In [ ]:
variants = engine.build_variants(cfg, raw, images)
val_curves = engine.curves(variants, splits, "val", cfg)
baseline = engine.random_baseline(splits, "val")
display(ShowImage(filename=str(export.plot_precision_curves(val_curves, None, baseline,
                                                            cfg.outputs_dir / "precision_curves.png"))))

## 6. Evaluation
Every variant on val and test. The deployed variant is the one with the best **validation**
Precision@10 (article type); test numbers are only reported. A random ranking scores the share of
catalog items that happen to share the query's label.

In [ ]:
results = engine.evaluate_all(variants, splits, cfg)
best = engine.select_best(results, cfg)
print("selected (validation):", best)
print("random-ranking Precision@K (test):", {k: round(v, 4) for k, v in engine.random_baseline(splits, "test").items()})
engine.comparison_table(results, cfg).round(4)

In [ ]:
print("max |sklearn NearestNeighbors - model.top_k| similarity:", engine.check_against_sklearn(variants[best]))
engine.per_category(variants[best], splits, cfg).round(4)

In [ ]:
display(ShowImage(filename=str(export.plot_comparison(results, list(engine.comparison_table(results, cfg).index), best,
                                                     cfg, cfg.outputs_dir / "variant_comparison.png"))))
display(ShowImage(filename=str(export.plot_queries(variants[best], splits, cfg.outputs_dir / "query_examples.png"))))

## 7. Inference
`export.save_model` writes the servable files into `cfg.model_dir` (`backbone.keras`, float16
`embeddings.npy`, `catalog.parquet` with thumbnails, `projection.npz` (PCA 1280->256 fit on the
catalog embeddings, when the deployed variant is a PCA one), `config.json`) and checks that `model.py` reproduces
the in-memory search. Then exactly what the Space runs: `model.load(dir, device).predict(image, k)`
on the Space's example photos (one held-out **test** photo per master category, never in the catalog).
**With `SMOKE_TEST=0` this overwrites `../model`** and `../space/examples`.

In [ ]:
train_time_s = time.perf_counter() - t0
export.save_model(cfg, embedder, variants[best], splits)
examples = data_setup.write_examples(splits.test, cfg.examples_dir, cfg.n_examples)

predictor = M.load(cfg.model_dir, "cpu")
for path in examples:
    result = predictor.predict(path, k=5)
    print(path.name)
    for m in result["matches"]:
        print(f"   {m['score']:.3f}  {m['master_category']:12s} {m['article_type']:15s} {m['name']}")

## 8. Export
`export.export` writes `metrics.json` (test numbers of the deployed variant, its per-master-category
P@5 under `per_category`, + every variant),
the card graphs in `assets/`, and refreshes the model card (`README.md`: metrics table, comparison table,
`ml_lab` block). To publish, set `PUSH_TO_HUB = True` in section 2. The token comes from `hf auth login`
(local) or an `HF_TOKEN` secret / environment variable (Colab: the Secrets panel). Smoke runs are never uploaded.

In [ ]:
metrics = export.export(cfg, results=results, best=best, variants=variants, splits=splits, val_curves=val_curves,
                        random_baseline=engine.random_baseline(splits, "test"), params=params,
                        device=utils.device_name("tensorflow"), embed_time_s=embed_time_s, train_time_s=train_time_s)
print({k: v for k, v in metrics.items() if k in ("model", "primary_metric", "metrics", "train_time_s")})
for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(f"{path.relative_to(cfg.model_dir).as_posix():30s} {path.stat().st_size / 1e6:8.2f} MB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))